In [4]:
import os
from getpass import getpass
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
import random


In [3]:
# Get Kaggle API Token
os.environ['KAGGLE_API_TOKEN'] = getpass("Kaggle API token: ")

# Download kaggle competition datasets
!kaggle competitions download -c llm-classification-finetuning

# Unzip the files
!unzip -q llm-classification-finetuning.zip -d data

# Lists the files
!ls -la data

100% 57.0M/57.0M [00:00<00:00, 110MB/s] 

total 179884
drwxr-xr-x 2 root root      4096 Jul 10 15:10 .
drwxr-xr-x 1 root root      4096 Jul 10 15:10 ..
-rw-r--r-- 1 root root       237 Oct  9  2024 sample_submission.csv
-rw-r--r-- 1 root root     10658 Oct  9  2024 test.csv
-rw-r--r-- 1 root root 184175071 Oct  9  2024 train.csv


In [2]:

#df = pd.read_csv("data/train.csv")
#df.head()

# Archetecure of model:

### input variables
```
prompt = prompt
response_1 = either reponse_a or response_b
response_2 = the other response_a/b
```
### Encoding model
```
model_1 = DeBERTA v3 base
```

### Classification model
```
model_2 = classification model
```
### pipeline of data
```
prompt -> model_1 -> output_p
response_1 -> model_1 -> output_1
response_2 -> model_1 -> output_2

(output_p, output_1, output_2) -> model_2 -> (prob of first response, prob of second response, prob of tie)
```



In [ ]:
class LLMModelsPref(Dataset):
    def __init__(self, prompts, responses_a, responses_b, labels, tokenizer, max_len = 512, prob = 0.5):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.prob = prob

        print(f"Processing {len(prompts)} prompts...")

        
        encoded_p = tokenizer(list(prompts), max_length = max_len, truncation = True, padding = "max_length", return_tensors = "pt")
        encoded_a = tokenizer(list(responses_a), max_length = max_len, truncation = True, padding = "max_length", return_tensors = "pt")
        encoded_b = tokenizer(list(responses_b), max_length = max_len, truncation = True, padding = "max_length", return_tensors = "pt")

        self.prompts = encoded_p["input_ids"]
        self.responses_a = encoded_a["input_ids"]
        self.responses_b = encoded_b["input_ids"]

        self.prompts_masks = encoded_p["attention_mask"]
        self.responses_a_masks = encoded_a["attention_mask"]
        self.responses_b_masks = encoded_b["attention_mask"]

        # Convert to LongTensor

        self.labels = torch.LongTensor(labels)

        print(f"Dataset created")

    def __len__(self):
        return len(self.prompts)
    
    def __getitem__(self, idx):
        prompt = self.prompts[idx]
        label = self.labels[idx]
        response_a = self.responses_a[idx]
        response_b = self.responses_b[idx]

        prompt_mask = self.prompts_masks[idx]
        response_a_mask = self.responses_a_masks[idx]
        response_b_mask = self.responses_b_masks[idx]

        if random.random() < self.prob:
            response_a, response_b = response_b, response_a
            response_a_mask, response_b_mask = response_b_mask, response_a_mask
            if label == 0:
                label = 1
            elif label ==1:
                label = 0
        return {
            "prompt": prompt, "prompt_mask": prompt_mask, 
            "response_a": response_a, "response_a_mask": response_a_mask, 
            "response_b": response_b, "response_b_mask": response_b_mask, 
            "label": label
        }
        

